In [34]:
import os
import time
import pickle
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sentence_transformers import SentenceTransformer, util
from transformers import CLIPProcessor, CLIPModel
import random


In [35]:
# Custom Dataset
class VQADataset(Dataset):
    def __init__(self, images_path, data_file, sentence_embeddings, data_split, train_ratio=1.0):
        self.images_path = images_path
        self.data_file = data_file
        self.sentence_embeddings = sentence_embeddings
        self.data_split = data_split.lower()
        self.train_ratio = train_ratio if self.data_split == "train" else 1.0

        self.image_data = []
        self.question_data = []
        self.answer_data = []
        self.question_embeddings_data = []
        self.answer_embeddings_data = []
        self.label_data = []
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        self.load_data()

    def load_data(self):
        print(f"LOADING data from {self.data_file}")
        random.seed(42)

        with open(self.data_file, 'r') as f:
            lines = f.readlines()

        if self.data_split == "train":
            random.shuffle(lines)
            num_samples = int(len(lines) * self.train_ratio)
            lines = lines[:num_samples]

        for line in lines:
            line = line.rstrip("\n")
            img_name, text, raw_label = line.split("\t")
            img_path = os.path.join(self.images_path, img_name.strip())

            question_answer_text = text.split("?")
            question_text = question_answer_text[0].strip() + '?'
            answers = question_answer_text[1].strip().split(" | ")

            # Create labels for each answer option (only one correct answer)
            correct_answer = answers[0].strip()  # Assuming the first answer is correct
            for i, answer in enumerate(answers):
                label = 1 if answer.strip() == correct_answer else 0
                self.image_data.append(img_path)
                self.question_data.append(question_text)
                self.answer_data.append(answer.strip())
                self.question_embeddings_data.append(self.sentence_embeddings[question_text])
                self.answer_embeddings_data.append(self.sentence_embeddings[answer.strip()])
                self.label_data.append(label)

        print(f"|image_data|={len(self.image_data)}")
        print(f"|question_data|={len(self.question_data)}")
        print(f"|answer_data|={len(self.answer_data)}")
        print("done loading data...")

    def __len__(self):
        return len(self.image_data)

    def __getitem__(self, idx):
        img_path = self.image_data[idx]
        img = Image.open(img_path).convert("RGB")
        img = self.transform(img)
        question_embedding = torch.tensor(self.question_embeddings_data[idx], dtype=torch.float32)
        answer_embedding = torch.tensor(self.answer_embeddings_data[idx], dtype=torch.float32)
        label = torch.tensor(self.label_data[idx], dtype=torch.long)

        # Check the shape of question_embedding
       # print(f"question_embedding shape: {question_embedding.shape}")

        return img, question_embedding, answer_embedding, label

In [36]:
def load_sentence_embeddings(file_path):
    print("READING sentence embeddings...")
    with open(file_path, 'rb') as f:
        data = pickle.load(f)
    return data

In [37]:
class Hybrid_CLIP_LSTM(nn.Module):
    def __init__(self, num_classes=4):
        super(Hybrid_CLIP_LSTM, self).__init__()
        # CLIP for vision and text
        self.clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
        self.clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
        
        # Freeze CLIP parameters except the last layer
        for name, param in self.clip_model.named_parameters():
            if "vision_model" in name:
                param.requires_grad = False
            elif "text_model" in name:
                param.requires_grad = False
            elif "proj" in name:
                param.requires_grad = True
        
        # LSTM for question processing
        self.lstm = nn.LSTM(input_size=768, hidden_size=128, num_layers=2, batch_first=True)
        
        # Fully connected layers
        self.fc1 = nn.Linear(512 + 128, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, images, question_embeddings, answer_embeddings):
        # Process image with CLIP
        image_features = self.clip_model.get_image_features(images)
        
        # Check the shape of question_embeddings
        #print(f"question_embeddings shape before processing: {question_embeddings.shape}")
        
        # Reshape question_embeddings to add sequence length dimension
        batch_size = question_embeddings.size(0)
        sequence_length = 1  # Assuming each question is a single vector
        question_embeddings = question_embeddings.view(batch_size, sequence_length, -1)
        
        # Check the shape after reshaping
        #print(f"question_embeddings shape after reshaping: {question_embeddings.shape}")
        
        # Process question with LSTM
        question_features, _ = self.lstm(question_embeddings)
        question_features = question_features[:, -1, :]  # Take last hidden state
        
        # Combine features
        combined_features = torch.cat((image_features, question_features), dim=1)
        
        # Classification
        x = torch.relu(self.fc1(combined_features))
        output = self.fc2(x)
        return output



In [38]:
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10):
    training_time = time.time()
    model.train()
    
    for epoch in range(num_epochs):
        running_loss = 0.0
        total_batches = len(train_loader)
        start_time = time.time()

        for batch_idx, (images, question_embeddings, answer_embeddings, labels) in enumerate(train_loader):
            images = images.to(device)          
            question_embeddings = question_embeddings.to(device) 
            answer_embeddings = answer_embeddings.to(device)  
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images, question_embeddings, answer_embeddings)

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

            if batch_idx % 100 == 0:
                print(f'Epoch [{epoch + 1}/{num_epochs}], Batch [{batch_idx}/{total_batches}], Loss: {loss.item():.4f}')

        avg_loss = running_loss / total_batches
        elapsed_time = time.time() - start_time
        print(f'Epoch [{epoch + 1}/{num_epochs}] Average Loss: {avg_loss:.4f}, {elapsed_time:.2f} seconds')

        # Validate the model on the validation set
        validate_model(model, val_loader, device)

    total_training_time = time.time() - training_time
    print(f'Total Training Time: {total_training_time:.2f} seconds')


In [39]:
def validate_model(model, val_loader, device):
    model.eval()
    total_correct = 0
    total_samples = 0
    mrr = 0.0
    em = 0.0

    with torch.no_grad():
        for images, question_embeddings, answer_embeddings, labels in val_loader:
            images = images.to(device)          
            question_embeddings = question_embeddings.to(device) 
            answer_embeddings = answer_embeddings.to(device)  
            labels = labels.to(device)  

            outputs = model(images, question_embeddings, answer_embeddings)
            predicted_probabilities = torch.softmax(outputs, dim=1)
            predicted_classes = predicted_probabilities.argmax(dim=1)

            total_samples += labels.size(0)
            total_correct += (predicted_classes == labels).sum().item()

            # Calculate MRR and EM
            sorted_indices = torch.argsort(predicted_probabilities, dim=1, descending=True)
            for i in range(len(labels)):
                rank = (sorted_indices[i] == labels[i]).nonzero().item() + 1
                mrr += 1.0 / rank
                em += 1.0 if rank == 1 else 0.0

    accuracy = total_correct / total_samples
    mrr /= total_samples
    em /= total_samples
    print(f'Validation Accuracy: {accuracy:.4f}, MRR: {mrr:.4f}, EM: {em:.4f}')
    model.train()

In [40]:
def evaluate_model(model, test_loader, device):
    model.eval()
    total_correct = 0
    total_samples = 0
    mrr = 0.0
    em = 0.0

    with torch.no_grad():
        for images, question_embeddings, answer_embeddings, labels in test_loader:
            images = images.to(device)          
            question_embeddings = question_embeddings.to(device) 
            answer_embeddings = answer_embeddings.to(device)  
            labels = labels.to(device)  

            outputs = model(images, question_embeddings, answer_embeddings)
            predicted_probabilities = torch.softmax(outputs, dim=1)
            predicted_classes = predicted_probabilities.argmax(dim=1)

            total_samples += labels.size(0)
            total_correct += (predicted_classes == labels).sum().item()

            # Calculate MRR and EM
            sorted_indices = torch.argsort(predicted_probabilities, dim=1, descending=True)
            for i in range(len(labels)):
                rank = (sorted_indices[i] == labels[i]).nonzero().item() + 1
                mrr += 1.0 / rank
                em += 1.0 if rank == 1 else 0.0

    accuracy = total_correct / total_samples
    mrr /= total_samples
    em /= total_samples
    print(f'Test Accuracy: {accuracy:.4f}, MRR: {mrr:.4f}, EM: {em:.4f}')

In [41]:
if __name__ == '__main__':
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f'Using device: {device}')

    IMAGES_PATH = "./visual7w-images"
    train_data_file = "./visual7w-text/v7w.TrainImages.itm.txt"
    val_data_file = "./visual7w-text/v7w.DevImages.itm.txt"
    test_data_file = "./visual7w-text/v7w.TestImages.itm.txt"
    sentence_embeddings_file = "./v7w.sentence_embeddings-gtr-t5-large.pkl"
    sentence_embeddings = load_sentence_embeddings(sentence_embeddings_file)

    train_dataset = VQADataset(IMAGES_PATH, train_data_file, sentence_embeddings, data_split="train", train_ratio=0.2)
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    
    val_dataset = VQADataset(IMAGES_PATH, val_data_file, sentence_embeddings, data_split="validation")
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
    
    test_dataset = VQADataset(IMAGES_PATH, test_data_file, sentence_embeddings, data_split="test")
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

    model = Hybrid_CLIP_LSTM(num_classes=4).to(device)
    print("\nModel Architecture:")
    print(model)

    # Print the parameters of the model
    total_params = 0
    print("\nModel Trainable Parameters:")
    for name, param in model.named_parameters():
        if param.requires_grad:
            num_params = param.numel()  
            total_params += num_params
            print(f"{name}: {param.data.shape} | Number of parameters: {num_params}")
    print(f"\nTotal number of parameters in the model: {total_params}")

    # Define loss function and optimizer 
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=1e-4)

    # Train and evaluate the model
    train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10)
    evaluate_model(model, test_loader, device)

Using device: cuda
READING sentence embeddings...
LOADING data from ./visual7w-text/v7w.TrainImages.itm.txt
|image_data|=9780
|question_data|=9780
|answer_data|=9780
done loading data...
LOADING data from ./visual7w-text/v7w.DevImages.itm.txt
|image_data|=6008
|question_data|=6008
|answer_data|=6008
done loading data...
LOADING data from ./visual7w-text/v7w.TestImages.itm.txt
|image_data|=5980
|question_data|=5980
|answer_data|=5980
done loading data...

Model Architecture:
Hybrid_CLIP_LSTM(
  (clip_model): CLIPModel(
    (text_model): CLIPTextTransformer(
      (embeddings): CLIPTextEmbeddings(
        (token_embedding): Embedding(49408, 512)
        (position_embedding): Embedding(77, 512)
      )
      (encoder): CLIPEncoder(
        (layers): ModuleList(
          (0-11): 12 x CLIPEncoderLayer(
            (self_attn): CLIPSdpaAttention(
              (k_proj): Linear(in_features=512, out_features=512, bias=True)
              (v_proj): Linear(in_features=512, out_features=512, bia